# Customize CLM Output

In this section, we will learn how to customize CLM history output. This includes:

* changing the output frequency
* controlling the number of time samples written to each history file
* adding extra variables to the output
* creating additional history files
* choosing whether added variables are time-averaged or instantaneous

These settings are controlled with three CLM namelist variables:

* `hist_nhtfrq`: output frequency of each history stream
* `hist_mfilt`: maximum number of time samples written to each history file
* `hist_fincl`: list of variables to add to each history stream

These variables should be added to `user_nl_clm`.


## CLM history file names: `h0a`, `h0i`, `h1a`, `h1i`, ...

CLM history files use a convention similar to CAM.

Averaged, accumulated, minimum, maximum, or sum fields are written to files with an `a` suffix:

```text
h0a, h1a, h2a, ...
```

Instantaneous fields are written to files with an `i` suffix:

```text
h0i, h1i, h2i, ...
```

For example, if `TSA` is written as an averaged field on the first history stream, it will appear in an `h0a` file. If `TSA:I` is requested as an instantaneous field on the second history stream, it will appear in an `h1i` file.


## Customizing CLM Output Frequency: `hist_nhtfrq`

The namelist variable `hist_nhtfrq` controls how often CLM writes history output.

* If `hist_nhtfrq = 0`, the file contains monthly averages.
* If `hist_nhtfrq > 0`, the output frequency is given as a number of model time steps.
* If `hist_nhtfrq < 0`, the output frequency is given as a number of hours.

For example:

```fortran
hist_nhtfrq = -24
```

means daily output, while:

```fortran
hist_nhtfrq = 24
```

means output every 24 model time steps.

<div class="alert alert-info">
<strong>Evaluate your understanding</strong>

How do you modify CLM output to write 3-hourly data?
</div>

<div class="alert alert-success">
<details>
<summary>Click here for the solution</summary>

Use:

```fortran
hist_nhtfrq = -3
```

This writes output every 3 hours.

</details>
</div>


## Adding Extra Variables and History Streams: `hist_fincl`

CLM can write multiple history streams. The history streams are numbered internally as:

```text
h0, h1, h2, ..., h9
```

The namelist variables `hist_fincl1`, `hist_fincl2`, ..., `hist_fincl10` control which extra variables are written to each stream.

* `hist_fincl1` controls fields added to the `h0` stream
* `hist_fincl2` controls fields added to the `h1` stream
* `hist_fincl3` controls fields added to the `h2` stream
* ...
* `hist_fincl10` controls fields added to the `h9` stream

For example, to add ground temperature, `TG`, to the default `h0` stream, add:

```fortran
hist_fincl1 = 'TG'
```

to `user_nl_clm`.

By default, added variables are usually written as time-averaged fields, so this would appear in an `h0a` file.

To request instantaneous output, add the `:I` flag:

```fortran
hist_fincl1 = 'TG:I'
```

This would appear in an `h0i` file.

## Customizing the Number of Samples per File: `hist_mfilt`

The namelist variable `hist_mfilt` controls the maximum number of time samples written to each history file.

For example:

```fortran
hist_mfilt = 10
```

means that each history file will contain up to 10 time samples.

To output daily data in a single file for a one-year run, use:

```fortran
hist_nhtfrq = -24
hist_mfilt  = 365
```

To output daily data with one time sample per file, use:

```fortran
hist_nhtfrq = -24
hist_mfilt  = 1
```


For monthly output, `hist_mfilt` is generally set to 1. Monthly CLM history files are normally written with one monthly average per file.


## Creating Additional History Streams

The first history stream, `h0`, is the default CLM history stream. You can create additional streams by setting values for `hist_fincl2`, `hist_fincl3`, and so on.

For example, to create a monthly `h1` stream with `TG` and `TSA`, use:

```fortran
hist_fincl2  = 'TG', 'TSA'
hist_nhtfrq  = 0, 0
hist_mfilt   = 1, 1
```

The first value in `hist_nhtfrq` and `hist_mfilt` applies to `h0`. The second value applies to `h1`.

Because `TG` and `TSA` are written as averaged fields here, they will appear in an `h1a` file.



<div class="alert alert-info">
<strong>Evaluate your understanding</strong>

What namelist settings would create:

* a monthly `h1` file with `TG` and `TSA`
* a daily `h2` file with `TG` and `TSA`
* 10 time samples in each `h1` and `h2` file
</div>

<div class="alert alert-success">
<details>
<summary>Click here for the solution</summary>

Use:

```fortran
hist_fincl2 = 'TG', 'TSA'
hist_fincl3 = 'TG', 'TSA'

hist_nhtfrq = 0, 0, -24
hist_mfilt  = 1, 10, 10
```

Explanation:

* `hist_fincl2` adds `TG` and `TSA` to the `h1` stream.
* `hist_fincl3` adds `TG` and `TSA` to the `h2` stream.
* The first value of `hist_nhtfrq` applies to `h0`.
* The second value applies to `h1`: monthly output.
* The third value applies to `h2`: daily output.
* The second and third values of `hist_mfilt` set 10 samples per file for `h1` and `h2`.

The files will be named using the averaged-file convention:

```text
h1a
h2a
```

</details>
</div>

<div class="alert alert-info">
<strong>Evaluate your understanding</strong>

What settings would generate CLM 3-hourly data for a month-long simulation, with one output file per day?
</div>

<div class="alert alert-success">
<details>
<summary>Click here for the solution</summary>

For 3-hourly output, there are 8 samples per day. To create one file per day, set `hist_mfilt = 8`.

For example, to write 3-hourly `TG` and `TSA` on the `h1` stream:

```fortran
hist_fincl2 = 'TG', 'TSA'

hist_nhtfrq = 0, -3
hist_mfilt  = 1, 8
```

Because no instantaneous flag is used, these fields will be written to an averaged file:

```text
h1a
```

</details>
</div>

## Averaging Flags for `hist_fincl` Variables

You can add an averaging flag after a variable name using a colon.

Common flags include:

* `A`: average
* `I`: instantaneous
* `M`: minimum
* `X`: maximum
* `SUM`: sum

For example:

```fortran
hist_fincl1 = 'TLAI:M'
```

adds the minimum value of `TLAI` to the `h0` stream.

Because `M` is not instantaneous, this field will appear in an `h0a` file.

To request instantaneous leaf area index, use:

```fortran
hist_fincl1 = 'TLAI:I'
```

This field will appear in an `h0i` file.

## Evaluate Your Understanding

What happens if we set the following in `user_nl_clm`?

```fortran
hist_fincl2 = 'TG',   'TSA'
hist_fincl3 = 'TG:I', 'TSA:I'
hist_fincl4 = 'TG',   'TSA'
hist_fincl5 = 'TG:I', 'TSA:I'

hist_nhtfrq = 0, -24, -24, -3, -3
hist_mfilt  = 1, 30, 30, 8, 8
```

<div class="alert alert-success">
<details>
<summary>Click here for the solution</summary>

This creates four additional CLM history streams:

* `h1a`: daily averaged `TG` and `TSA`
* `h2i`: daily instantaneous `TG` and `TSA`
* `h3a`: 3-hourly averaged `TG` and `TSA`
* `h4i`: 3-hourly instantaneous `TG` and `TSA`

The `hist_mfilt` settings mean:

* `h1a` contains 30 daily samples per file
* `h2i` contains 30 daily samples per file
* `h3a` contains 8 three-hourly samples per file, or one day per file
* `h4i` contains 8 three-hourly samples per file, or one day per file

</details>
</div>


Increasing output frequency or adding many variables can substantially increase the amount of data written and can slow down the model. Add only the variables and frequencies needed for your exercise.

